# RSNA Knee Abnormality Detection — Image Baseline (skeleton)

This notebook is meant to run **on Kaggle**, not locally — the competition data
(247 GB of DICOM images) is mounted at `/kaggle/input/rsna-knee-abnormality-detection/`
on Kaggle's servers, and is not available on this machine.

Goal of this skeleton: get a full pipeline running end-to-end without errors —
load images, train a small CNN, predict, produce a valid `submission.csv` — as a
foundation to iterate on. It is **not** expected to score well yet:
- Only 58 of 4407 training studies currently have labels.
- We start from a single mid-sagittal slice per study (simplest possible input);
  extending to multi-slice / 3D / multi-plane fusion is the natural next step.

Constraints from the competition rules: <= 9h runtime, internet disabled at
submission time (so pretrained weights must come from Kaggle's local cache or a
Kaggle Dataset input, not a live download).

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

def pick_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        major, minor = torch.cuda.get_device_capability(0)
        cap_str = f"sm_{major}{minor}"
        if cap_str in torch.cuda.get_arch_list():
            return torch.device("cuda")
        print(f"GPU capability {cap_str} unsupported by this PyTorch build "
              f"(supports {torch.cuda.get_arch_list()}); falling back to CPU.")
    except Exception as e:
        print(f"GPU capability check failed ({e}); falling back to CPU.")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)

In [ ]:
COMPETITION_SLUG = "rsna-knee-abnormality-detection"
DATA_DIR = Path(f"/kaggle/input/competitions/{COMPETITION_SLUG}")

TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_SERIES_CSV = DATA_DIR / "train_series.csv"
TEST_CSV = DATA_DIR / "test.csv"
TEST_SERIES_CSV = DATA_DIR / "test_series.csv"
TRAIN_SERIES_DIR = DATA_DIR / "train_series"
TEST_SERIES_DIR = DATA_DIR / "test_series"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

IMG_SIZE = 224
N_SLICES = 5  # evenly spaced slices per study, instead of just the middle one
RANDOM_STATE = 42
BATCH_SIZE = 4  # x N_SLICES images per study now, so a smaller batch keeps memory sane
N_EPOCHS = 5  # skeleton default — increase once the pipeline is validated

In [ ]:
def make_folds(n_rows: int, n_splits: int = 5, random_state: int = RANDOM_STATE):
    """Same logic as src/folds.py's make_folds — kept in sync by hand since this
    notebook can't import from the repo. Must produce identical fold assignments
    to the text baseline (same KFold params, same row order) so OOF predictions
    from both models can be ensembled study-for-study."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    folds = np.zeros(n_rows, dtype=int)
    for fold_id, (_, val_idx) in enumerate(kf.split(np.arange(n_rows))):
        folds[val_idx] = fold_id
    return folds

## Data loading

For each study we pick one series to keep things simple: prefer a **sagittal**
series (ACL/menisci are best seen sagittally), otherwise fall back to whatever
series is available. From that series we now take **N_SLICES evenly spaced
slices** (instead of a single middle slice) — a series typically has 20-45
slices, so one slice was discarding most of the study's information.

In [ ]:
def load_labels():
    df = pd.read_csv(TRAIN_CSV)
    return df.dropna(subset=LABELS).reset_index(drop=True)


def pick_series(study_uid: str, series_df: pd.DataFrame) -> str | None:
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    if rows.empty:
        return None
    sagittal = rows[rows["Anatomical_Plane"] == "Sagittal"]
    chosen = sagittal.iloc[0] if len(sagittal) else rows.iloc[0]
    return chosen["SeriesInstanceUID"]


def _load_dicom_array(path: Path) -> np.ndarray:
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr  # shape (H, W), values in [0, 1]


def load_slices(series_dir: Path, n_slices: int = N_SLICES) -> list[np.ndarray]:
    """N_SLICES evenly spaced slices across the series (by DICOM InstanceNumber
    order), instead of just the single middle slice — a series typically has
    20-45 slices, so a single one discards most of the study's information."""
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        raise FileNotFoundError(f"No .dcm files in {series_dir}")

    def instance_number(path):
        try:
            return pydicom.dcmread(path, stop_before_pixels=True).InstanceNumber
        except Exception:
            return 0

    dcm_files = sorted(dcm_files, key=instance_number)
    n = len(dcm_files)
    if n <= n_slices:
        indices = list(range(n)) + [n - 1] * (n_slices - n)  # pad by repeating the last slice
    else:
        indices = np.linspace(0, n - 1, n_slices).round().astype(int).tolist()
    return [_load_dicom_array(dcm_files[i]) for i in indices]

In [ ]:
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class KneeMRIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, series_dir: Path, has_labels: bool):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.series_dir = series_dir
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = row["StudyInstanceUID"]
        series_uid = pick_series(study_uid, self.series_df)
        slices = load_slices(self.series_dir / study_uid / series_uid)
        tensors = []
        for img in slices:
            img_rgb = np.stack([img, img, img], axis=-1)  # fake 3-channel for pretrained CNN
            tensors.append(preprocess(img_rgb))
        stacked = torch.stack(tensors)  # (N_SLICES, 3, H, W)

        if self.has_labels:
            y = torch.tensor(row[LABELS].values.astype(np.float32))
            return stacked, y
        return stacked, study_uid

## Model

A ResNet18 backbone, shared across all N_SLICES slices of a study: each slice
is passed through the same convolutional features, then the per-slice
pooled features are **averaged** before the final 12-way sigmoid head — a
simple multi-instance approach that stays compatible with 2D ImageNet
pretraining (no need to touch the first conv layer's channel count).

We try to load ImageNet pretrained weights; if internet is disabled and they
aren't already cached in the Kaggle image, we fall back to random init so the
pipeline doesn't crash.

In [ ]:
RESNET18_WEIGHTS_PATH = Path("/kaggle/input/resnet18-imagenet-weights/resnet18-f37072fd.pth")


class MultiSliceResNet(nn.Module):
    def __init__(self, n_labels: int = len(LABELS)):
        super().__init__()
        backbone = models.resnet18(weights=None)
        if RESNET18_WEIGHTS_PATH.exists():
            state_dict = torch.load(RESNET18_WEIGHTS_PATH, map_location="cpu")
            backbone.load_state_dict(state_dict)
            print("Loaded ImageNet-pretrained ResNet18 weights from local dataset.")
        else:
            print(f"{RESNET18_WEIGHTS_PATH} not found; using random init.")
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # everything up to avgpool
        self.n_features = backbone.fc.in_features
        self.fc = nn.Linear(self.n_features, n_labels)

    def forward(self, x):
        # x: (batch, N_SLICES, 3, H, W)
        b, n, c, h, w = x.shape
        x = x.view(b * n, c, h, w)
        feats = self.features(x).view(b, n, self.n_features)
        feats = feats.mean(dim=1)  # average pooled features across slices
        return self.fc(feats)


def build_model(n_labels: int = len(LABELS)) -> nn.Module:
    return MultiSliceResNet(n_labels)

## Training

5-fold cross-validation over the 58 labeled studies, using the **same fold
assignment as the text baseline** (`make_folds` above, mirrored from
`src/folds.py`) so the two models' out-of-fold predictions can be ensembled
per study later. Out-of-fold predictions are saved to `image_oof.csv`. We
then re-fit on *all* labeled data for the model used at test time.

In [ ]:
def train_one_fold(train_ds, val_ds=None):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    model = build_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(N_EPOCHS):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        print(f"Epoch {epoch + 1}/{N_EPOCHS} — train loss: {total_loss / len(train_ds):.4f}")

    if val_ds is None:
        return model, None, None

    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            probs = torch.sigmoid(model(x)).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(y.numpy())
    return model, np.concatenate(all_preds), np.concatenate(all_targets)


def macro_auc(preds: np.ndarray, targets: np.ndarray) -> float:
    scores = []
    for i, label in enumerate(LABELS):
        if len(np.unique(targets[:, i])) < 2:
            continue
        scores.append(roc_auc_score(targets[:, i], preds[:, i]))
        print(f"  {label:<20} AUC={scores[-1]:.3f}")
    macro = float(np.mean(scores)) if scores else float("nan")
    print(f"Macro AUC: {macro:.3f}")
    return macro

In [ ]:
labeled_df = load_labels()
series_df = pd.read_csv(TRAIN_SERIES_CSV)
print(f"Labeled training studies: {len(labeled_df)}")

folds = make_folds(len(labeled_df))
oof_preds = np.full((len(labeled_df), len(LABELS)), 0.5)

for fold_id in np.unique(folds):
    print(f"\n=== Fold {fold_id} ===")
    train_mask = folds != fold_id
    val_mask = folds == fold_id
    train_ds = KneeMRIDataset(labeled_df[train_mask], series_df, TRAIN_SERIES_DIR, has_labels=True)
    val_ds = KneeMRIDataset(labeled_df[val_mask], series_df, TRAIN_SERIES_DIR, has_labels=True)
    _, val_preds, _ = train_one_fold(train_ds, val_ds)
    oof_preds[val_mask] = val_preds

oof_df = pd.DataFrame(oof_preds, columns=LABELS)
oof_df.insert(0, "StudyInstanceUID", labeled_df["StudyInstanceUID"].values)
oof_df.to_csv("image_oof.csv", index=False)
print("\nSaved out-of-fold predictions to image_oof.csv")

print("\nOut-of-fold AUC (pooled across folds):")
macro_auc(oof_preds, labeled_df[LABELS].values.astype(int))

# Re-fit on all labeled data for the model used at test time (no more held-out val set).
full_ds = KneeMRIDataset(labeled_df, series_df, TRAIN_SERIES_DIR, has_labels=True)
final_model, _, _ = train_one_fold(full_ds)

## Inference on the test set

Builds `submission.csv` in the exact format expected by the leaderboard.

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
test_ds = KneeMRIDataset(test_df, test_series_df, TEST_SERIES_DIR, has_labels=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

final_model.eval()
uids, preds = [], []
with torch.no_grad():
    for x, study_uid in test_loader:
        x = x.to(DEVICE)
        probs = torch.sigmoid(final_model(x)).cpu().numpy()
        uids.extend(study_uid)
        preds.append(probs)

preds = np.concatenate(preds)
submission = pd.DataFrame(preds, columns=LABELS)
submission.insert(0, "StudyInstanceUID", uids)
submission.to_csv("submission.csv", index=False)
submission.head()